# Lab 5.7: Session State & Resumption

**What you'll build:** A session management system that handles stale context correctly -- and learn when to resume vs start fresh.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- resume a stale session and see the agent make wrong assumptions | 5 min |
| 2 | The Right Way -- fresh session + structured summary avoids stale context | 10 min |
| 3 | Your Turn -- build a session summary for handoff | 5 min |
| 4 | Stress Test -- decide resume vs fresh for multiple scenarios | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You have a code review agent that analyzed a pull request yesterday. Overnight, the developer pushed fixes for some issues and added new code. The question: should you resume the old session or start fresh?

We'll simulate this with two snapshots of the same codebase.

In [ ]:
# Snapshot 1: The code as it was YESTERDAY (when the agent first reviewed it)
CODE_V1 = {
    "auth.py": '''def authenticate(username, password):
    """Authenticate user with username and password."""
    user = db.find_user(username)
    if user and user.password == password:  # BUG: plaintext comparison
        return create_token(user)
    return None
''',
    "api.py": '''def get_user_data(token):
    """Get user data. No rate limiting."""
    user = validate_token(token)
    return db.get_all_user_data(user.id)  # BUG: returns ALL data, no filtering
'''
}

# Snapshot 2: The code as it is TODAY (developer pushed fixes + new code)
CODE_V2 = {
    "auth.py": '''def authenticate(username, password):
    """Authenticate user with username and password."""
    user = db.find_user(username)
    if user and verify_hash(password, user.password_hash):  # FIXED: uses hash comparison
        return create_token(user)
    return None
''',
    "api.py": '''def get_user_data(token):
    """Get user data with rate limiting."""
    user = validate_token(token)
    if not rate_limiter.check(user.id):
        raise RateLimitError("Too many requests")
    return db.get_user_profile(user.id)  # FIXED: returns profile only
''',
    "payments.py": '''def process_payment(user_id, amount):
    """Process payment. NEW FILE added overnight."""
    if amount <= 0:
        raise ValueError("Invalid amount")
    # BUG: No authentication check -- anyone with a user_id can charge
    return payment_gateway.charge(user_id, amount)
'''
}

# The agent's findings from yesterday's session
YESTERDAY_FINDINGS = [
    {"file": "auth.py", "issue": "Plaintext password comparison", "severity": "critical", "status": "open"},
    {"file": "api.py", "issue": "No rate limiting on user data endpoint", "severity": "high", "status": "open"},
    {"file": "api.py", "issue": "Returns all user data without filtering", "severity": "high", "status": "open"},
]

print("Yesterday's code: 2 files")
print("Today's code: 3 files (1 new, 2 updated)")
print(f"Yesterday's findings: {len(YESTERDAY_FINDINGS)}")
for f in YESTERDAY_FINDINGS:
    print(f"  [{f['severity'].upper()}] {f['file']}: {f['issue']}")

---

## Phase 1: The Wrong Approach -- Resume Stale Session

Simulate resuming the old session. The agent still thinks it's looking at yesterday's code.

In [ ]:
# Simulate resumed session: agent has yesterday's context
resumed_messages = [
    {"role": "user", "content": f"Review this code for security issues:\n\n" + 
     "\n".join(f"--- {f} ---\n{c}" for f, c in CODE_V1.items())},
    {"role": "assistant", "content": f"""I found {len(YESTERDAY_FINDINGS)} security issues:

1. **CRITICAL** auth.py: Plaintext password comparison - the code compares passwords directly instead of using hash comparison.
2. **HIGH** api.py: No rate limiting - the get_user_data endpoint has no protection against abuse.
3. **HIGH** api.py: Data over-exposure - returns all user data without filtering sensitive fields.

All three issues need to be addressed before merging."""},
    # Now the user "resumes" and asks to continue
    {"role": "user", "content": "Continue your review. Are there any other issues you missed?"}
]

print("=== RESUMED (STALE) SESSION ===")
print("The agent still sees YESTERDAY's code in its context...\n")

resumed_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=resumed_messages
)

print(f"Agent's response:\n{resumed_response.content[0].text}")

print("\n=== PROBLEMS WITH RESUMED SESSION ===")
print("1. Agent does NOT know auth.py was fixed (still thinks plaintext comparison exists)")
print("2. Agent does NOT know api.py was fixed (still thinks no rate limiting)")
print("3. Agent does NOT know payments.py exists (new file added overnight)")
print("4. Agent may CONFIRM its prior findings are still valid (but they're not!)")

---

## Phase 2: The Right Approach -- Fresh Session + Summary

Start a new session with a structured summary of prior work and the CURRENT code.

In [ ]:
# Build structured summary of prior session
session_summary = {
    "prior_review": {
        "date": "yesterday",
        "files_reviewed": list(CODE_V1.keys()),
        "findings": YESTERDAY_FINDINGS
    },
    "changes_since": {
        "updated_files": ["auth.py", "api.py"],
        "new_files": ["payments.py"],
        "deleted_files": []
    }
}

fresh_prompt = f"""You are continuing a security code review. A prior review was done yesterday,
but the code has been updated since then. Review the CURRENT code.

PRIOR REVIEW SUMMARY:
{json.dumps(session_summary, indent=2)}

CURRENT CODE (as of today):
"""

for filename, code in CODE_V2.items():
    fresh_prompt += f"\n--- {filename} ---\n{code}\n"

fresh_prompt += """
TASKS:
1. For each prior finding, check if it was fixed in the current code
2. Review new/updated files for NEW security issues
3. Report: which prior issues are fixed, which remain, and any new issues

Return a structured JSON response:
{"fixed": [...], "still_open": [...], "new_issues": [...]}
Each item: {"file": "...", "issue": "...", "severity": "...", "details": "..."}
Return ONLY the JSON."""

print("=== FRESH SESSION + SUMMARY ===")

fresh_response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    messages=[{"role": "user", "content": fresh_prompt}]
)

raw = fresh_response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    fresh_findings = json.loads(raw)
except json.JSONDecodeError:
    fresh_findings = {"raw": raw}

print(json.dumps(fresh_findings, indent=2))

In [ ]:
# Validate the fresh session results
print("=== VALIDATION ===")

expected_fixed = ["auth.py plaintext", "api.py rate limiting", "api.py data exposure"]
expected_new = ["payments.py no auth check"]

fixed_items = fresh_findings.get("fixed", [])
new_items = fresh_findings.get("new_issues", [])
still_open = fresh_findings.get("still_open", [])

print(f"Fixed issues found: {len(fixed_items)}")
for f in fixed_items:
    print(f"  [FIXED] {f.get('file', '?')}: {f.get('issue', '?')[:80]}")

print(f"\nNew issues found: {len(new_items)}")
for f in new_items:
    print(f"  [NEW] {f.get('file', '?')}: {f.get('issue', '?')[:80]}")

print(f"\nStill open: {len(still_open)}")
for f in still_open:
    print(f"  [OPEN] {f.get('file', '?')}: {f.get('issue', '?')[:80]}")

# Check correctness
found_fixed = len(fixed_items) >= 2  # At least auth.py and api.py fixes
found_new_payment_bug = any("payment" in str(f).lower() for f in new_items)

print(f"\n{'='*50}")
print(f"Detected fixes from updated code: {'YES' if found_fixed else 'NO'}")
print(f"Found new payment.py bug: {'YES' if found_new_payment_bug else 'NO'}")

if found_fixed and found_new_payment_bug:
    print("\nFresh session + summary correctly handled the updated codebase.")
    print("It recognized fixes AND found new issues the resumed session would have missed.")

---

## Phase 3: Your Turn -- Build a Session Summary

Design a structured session summary for a different scenario.

In [ ]:
# =============================================================
# TODO: Build a session summary for this scenario:
# =============================================================
#
# An agent spent 30 minutes debugging a production issue.
# It found that:
# - The error is a timeout in the payment service
# - It tried restarting the service (didn't help)
# - It checked the database (connections are normal)
# - It suspects the payment gateway API is slow
# - It has NOT checked the payment gateway status page yet
#
# The session was interrupted. The underlying system has NOT changed.
# Question: Should you resume or start fresh?

debug_summary = {
    # TODO: Fill in the summary
    # "situation": "...",
    # "key_findings": [ ... ],
    # "approaches_tried": [ {"action": "...", "result": "..."} ],
    # "current_hypothesis": "...",
    # "remaining_steps": [ ... ],
    # "recommendation": "resume" or "fresh_session"
}

print("Debug session summary:")
print(json.dumps(debug_summary, indent=2))

In [ ]:
# Checker
print("=== SESSION SUMMARY CHECKLIST ===")

checks = [
    ("Has key findings", "key_findings" in debug_summary and len(debug_summary.get("key_findings", [])) > 0),
    ("Has approaches tried", "approaches_tried" in debug_summary and len(debug_summary.get("approaches_tried", [])) > 0),
    ("Has remaining steps", "remaining_steps" in debug_summary and len(debug_summary.get("remaining_steps", [])) > 0),
    ("Has recommendation", "recommendation" in debug_summary),
    ("Recommends resume (system unchanged)", debug_summary.get("recommendation") == "resume"),
]

passed = sum(1 for _, r in checks if r)
for label, result in checks:
    print(f"  [{'PASS' if result else 'FAIL'}] {label}")

if passed == len(checks):
    print(f"\nPASSED -- Good summary. Resume is correct because the system has not changed.")
else:
    print(f"\n{passed}/{len(checks)} checks passed.")
    if debug_summary.get("recommendation") != "resume":
        print("Hint: The system has NOT changed, so --resume is appropriate here.")

---

## Phase 4: Decision Framework

In [ ]:
print("=== RESUME vs FRESH SESSION DECISION FRAMEWORK ===")
print()

scenarios = [
    ("Code review; developer pushed fixes overnight", "Fresh + summary", "Code has changed -- stale context"),
    ("Debug session; internet dropped; system unchanged", "Resume", "System state is the same"),
    ("Data analysis; database migrated since last session", "Fresh + summary", "Data has changed"),
    ("Long research task; interrupted; sources unchanged", "Resume", "No external changes"),
    ("Security audit; 15 PRs merged since last session", "Fresh + summary", "Codebase has changed"),
    ("Writing task; user wants to continue from draft", "Resume", "No external state to go stale"),
]

print(f"{'Scenario':<55} {'Decision':<20} {'Reason'}")
print(f"{'-'*55} {'-'*20} {'-'*40}")
for scenario, decision, reason in scenarios:
    print(f"{scenario:<55} {decision:<20} {reason}")

print("\n=== THE RULE ===")
print("Has the underlying data/code/system CHANGED since the session was paused?")
print("  YES -> Fresh session + structured summary of prior work")
print("  NO  -> Resume is safe and efficient")

### Key Takeaways

1. **--resume restores context but not reality.** If the underlying data changed, the session operates on stale assumptions.
2. **Fresh session + summary avoids stale context.** Start new, pass what was found before, and let the agent re-examine current state.
3. **Resume is fine when nothing changed.** If the system state is identical, resuming is efficient and correct.
4. **Session summaries should be structured.** Include: findings, approaches tried, remaining steps. Exclude: full conversation history.